# Build DPO Hard-Sample Dataset (vLLM, Qwen3.5-4B, answer-last)

## 1. Install dependencies

In [ ]:
!pip install -q \
  "vllm==0.17.0" \
  "huggingface_hub>=0.30,<0.36" \
  pillow

!pip install -q --no-deps --upgrade "tokenizers>=0.22"
!pip install -q --no-deps "transformers==5.3.0"

!pip install -q --no-deps --force-reinstall "huggingface_hub>=1.3.0,<2.0"
!pip install -q --no-deps --force-reinstall "tokenizers==0.22.2"

import importlib.metadata as md
for pkg in ["vllm", "transformers", "tokenizers", "huggingface_hub"]:
    try:
        print(f"{pkg:18s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:18s} NOT INSTALLED")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. HuggingFace login & 모델 선택

In [ ]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
MODEL_ID = 'minsu0567/IAD-X1-SFT-answer-last'

login(token=HF_TOKEN)
print('HuggingFace login OK.')
print('MODEL_ID:', MODEL_ID)

## 4. GPU check

In [ ]:
!nvidia-smi

## 5. Paths

In [ ]:
import os, json

DRIVE_ROOT     = '/content/drive/MyDrive'
GRPO_DIR       = f'{DRIVE_ROOT}/GRPO_dataset3'
DPO_SRC        = f'{DRIVE_ROOT}/IAD-X1/dpo_src'
INPUT_JSON     = f'{GRPO_DIR}/grpo_merged.json'
OUTPUT_JSON    = f'{DRIVE_ROOT}/dpo_hard_samples_answer_last.json'
LOCAL_JSON     = '/content/dpo_hard_samples_answer_last.json'
BUILD_SCRIPT   = f'{DPO_SRC}/build_dpo_dataset_qwen3_5_answer_last.py'

assert os.path.isdir(GRPO_DIR),      f'Missing directory: {GRPO_DIR}'
assert os.path.isfile(INPUT_JSON),   f'Missing: {INPUT_JSON}'
assert os.path.isfile(BUILD_SCRIPT), f'Missing: {BUILD_SCRIPT}'

with open(INPUT_JSON, encoding='utf-8') as f:
    _samples = json.load(f)
_bad = [i for i, s in enumerate(_samples) if not {'image', 'problem', 'solution'} <= set(s)]
assert not _bad, f'Unexpected schema at indices {_bad[:5]}'

print('Input :', INPUT_JSON, f'({len(_samples)} samples)')
print('Output:', OUTPUT_JSON)
print('Script:', BUILD_SCRIPT)

## 6. Debug — 앞 3개 raw 출력 확인

In [ ]:
!python {BUILD_SCRIPT} \
  --model-id "{MODEL_ID}" \
  --hf-token "{HF_TOKEN}" \
  --input-json "{INPUT_JSON}" \
  --output-json "{OUTPUT_JSON}" \
  --resize-to 512 \
  --max-model-len 8192 \
  --max-tokens 1024 \
  --gpu-mem 0.9 \
  --debug-n 3 \
  --debug-only

## 7. Build DPO dataset

In [ ]:
!python {BUILD_SCRIPT} \
  --model-id "{MODEL_ID}" \
  --hf-token "{HF_TOKEN}" \
  --input-json "{INPUT_JSON}" \
  --output-json "{OUTPUT_JSON}" \
  --local-json "{LOCAL_JSON}" \
  --batch-size 32 \
  --checkpoint-every 10 \
  --resize-to 512 \
  --max-model-len 8192 \
  --max-tokens 1024 \
  --gpu-mem 0.9 \
  --debug-n 3